# Calculus and gradients

Training a model means moving parameters downhill on a loss surface. The gradient tells you which way is downhill. This notebook covers derivatives, partial derivatives, the gradient, the chain rule and the Jacobian, then runs gradient descent on a 2D surface.

Here, 'partial differential' means partial derivatives and gradients as used in ML, not PDE solving.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

rng = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Intuition

A derivative is the slope of a curve at one point. If $f'(3) = 6$, nudging $x$ by 0.01 changes $f$ by about 0.06.

With several inputs, a partial derivative is the slope along one axis while the others stay fixed. Stack them and you get the gradient, an arrow that points uphill in the steepest direction. Step the other way and the loss falls.

## 2. Maths

Derivative: $f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$.

Gradient of $f:\mathbb{R}^n \to \mathbb{R}$: $\nabla f = \left[\frac{\partial f}{\partial x_1}, \dots, \frac{\partial f}{\partial x_n}\right]$.

Steepest ascent: the directional derivative along a unit vector $u$ is $\nabla f \cdot u = \lVert \nabla f \rVert \cos\theta$. It is largest when $u$ points along $\nabla f$.

Chain rule: if $z = f(g(x))$ then $\frac{dz}{dx} = f'(g(x)) \, g'(x)$. For a chain of functions, multiply the local derivatives.

Jacobian of $f:\mathbb{R}^n \to \mathbb{R}^m$: the $m \times n$ matrix $J_{ij} = \partial f_i / \partial x_j$. For compositions, $J_{f \circ g} = J_f J_g$. The chain rule is a matrix product.

Gradient descent: $\theta \leftarrow \theta - \eta \nabla L(\theta)$ with learning rate $\eta$.

## 3. Cost

One gradient costs about as much as 2 to 3 forward passes when computed analytically (backprop). A numerical gradient costs $2n$ forward passes for $n$ parameters. For a model with 1,000,000 weights that is 2,000,000 passes against about 3. That gap is why backprop exists, and why numerical gradients are only used as a check.

## 4. From scratch

### Derivative by finite differences

In [ ]:
def f(x):
    return x**3 - 2 * x


def numeric_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)


x0 = 3.0
print("numeric:", numeric_derivative(f, x0), " analytic 3x^2 - 2:", 3 * x0**2 - 2)

### Partial derivatives and the gradient

Use $f(x, y) = x^2 + 3y^2 + xy$. The analytic gradient is $[2x + y,\; 6y + x]$.

In [ ]:
def surface(p):
    x, y = p
    return x**2 + 3 * y**2 + x * y


def surface_grad(p):
    x, y = p
    return np.array([2 * x + y, 6 * y + x])


def numeric_grad(fn, p, h=1e-5):
    p = np.asarray(p, dtype=float)
    g = np.zeros_like(p)
    for i in range(p.size):
        step = np.zeros_like(p)
        step[i] = h
        g[i] = (fn(p + step) - fn(p - step)) / (2 * h)
    return g


p = np.array([1.0, 2.0])
print("analytic:", surface_grad(p))
print("numeric: ", numeric_grad(surface, p))

### Gradient as steepest ascent

Try 360 unit directions from one point and measure how fast $f$ rises along each. The best one should line up with the gradient.

In [ ]:
angles = np.linspace(0, 2 * np.pi, 360, endpoint=False)
dirs = np.column_stack([np.cos(angles), np.sin(angles)])
h = 1e-4
rise = np.array([(surface(p + h * u) - surface(p)) / h for u in dirs])

best = dirs[rise.argmax()]
g = surface_grad(p)
print("best direction:", best)
print("gradient direction:", g / np.linalg.norm(g))
print("max rise:", rise.max(), " gradient norm:", np.linalg.norm(g))

The fastest rise equals $\lVert \nabla f \rVert$, and it happens along the gradient. Going the opposite way gives the fastest fall.

### Chain rule

Take $L = (wx + b - y)^2$, the squared error of one prediction. Let $u = wx + b - y$. Then $\partial L/\partial w = 2u \cdot x$ and $\partial L/\partial b = 2u$.

In [ ]:
x_, y_ = 2.0, 4.0
loss = lambda wb: (wb[0] * x_ + wb[1] - y_) ** 2
wb = np.array([1.0, 0.0])
u = wb[0] * x_ + wb[1] - y_
manual = np.array([2 * u * x_, 2 * u])
print("chain rule:", manual, " numeric:", numeric_grad(loss, wb))

# sympy confirms it symbolically
w, b = sp.symbols("w b")
L = (w * x_ + b - y_) ** 2
print("sympy:", [sp.diff(L, v).subs({w: 1.0, b: 0.0}) for v in (w, b)])

### Jacobian

For $f(x, y) = [x^2 y,\; 5x + \sin y]$ the Jacobian is a 2 by 2 matrix. We build it from numerical gradients, one row per output, and compare with the analytic form.

In [ ]:
def vec_fn(p):
    x, y = p
    return np.array([x**2 * y, 5 * x + np.sin(y)])


def numeric_jacobian(fn, p, h=1e-5):
    p = np.asarray(p, dtype=float)
    cols = []
    for i in range(p.size):
        step = np.zeros_like(p)
        step[i] = h
        cols.append((fn(p + step) - fn(p - step)) / (2 * h))
    return np.column_stack(cols)


def analytic_jacobian(p):
    x, y = p
    return np.array([[2 * x * y, x**2], [5.0, np.cos(y)]])


q = np.array([1.5, 0.5])
print(numeric_jacobian(vec_fn, q))
print(analytic_jacobian(q))

### Gradient descent on a 2D surface

Minimise the same bowl $f(x, y) = x^2 + 3y^2 + xy$. The minimum is at the origin.

In [ ]:
def descend(grad_fn, start, lr, steps):
    path = [np.asarray(start, dtype=float)]
    for _ in range(steps):
        path.append(path[-1] - lr * grad_fn(path[-1]))
    return np.array(path)


path = descend(surface_grad, [4.0, 3.0], lr=0.1, steps=40)
print("start loss:", surface(path[0]), "end loss:", surface(path[-1]))
print("end point:", path[-1])

gx, gy = np.meshgrid(np.linspace(-5, 5, 100), np.linspace(-4, 4, 100))
Z = gx**2 + 3 * gy**2 + gx * gy
fig, ax = plt.subplots(figsize=(5, 4))
ax.contour(gx, gy, Z, levels=20, alpha=0.6)
ax.plot(path[:, 0], path[:, 1], "r.-")
ax.set_title("gradient descent, lr = 0.1")
plt.show()

## 5. Library version

SymPy differentiates symbolically and `lambdify` turns the result into a fast NumPy function.

In [ ]:
xs, ys = sp.symbols("x y")
expr = xs**2 + 3 * ys**2 + xs * ys
grad_sym = [sp.diff(expr, v) for v in (xs, ys)]
print("symbolic gradient:", grad_sym)
grad_fn = sp.lambdify((xs, ys), grad_sym)
print("at (1, 2):", grad_fn(1.0, 2.0))

# the Jacobian too
F = sp.Matrix([xs**2 * ys, 5 * xs + sp.sin(ys)])
print(F.jacobian([xs, ys]))

## 6. What breaks

The learning rate decides everything. Too small is slow. Too large overshoots, and past a threshold the loss grows without bound. For this bowl the largest eigenvalue of the Hessian is about 6.2, so the stable limit is roughly $2/6.2 \approx 0.32$.

In [ ]:
for lr in (0.01, 0.1, 0.3, 0.35):
    p_end = descend(surface_grad, [4.0, 3.0], lr=lr, steps=40)[-1]
    print(f"lr {lr:5.2f}: final loss {surface(p_end):.3e}")

Finite-difference step size also matters. Too big and truncation error dominates. Too small and float rounding takes over.

In [ ]:
for h in (1e-1, 1e-3, 1e-5, 1e-8, 1e-12):
    err = abs(numeric_derivative(f, 3.0, h) - 25.0)
    print(f"h = {h:.0e}: error {err:.2e}")

Around $10^{-5}$ is the sweet spot for float64 with central differences. The next notebook builds the exact method that avoids this trade-off.